# Notebook 03 - Model Analysis

## Evaluation contract and scope

Analyze completed baseline runs from persisted run artifacts in `outputs/runs`.

- Primary metric: ROC-AUC. Guardrails: PR-AUC and LogLoss.
- Split protocol: 5-fold x 3-repeat stratified cross-validation.
- Verification: assert comparability of split and data hashes across runs.
- Execution mode: read persisted artifacts only; no training or HPO executed in notebook cells.

Every numeric claim below is produced by the cell directly above it.

## 1. Artifact compatibility audit

Load all metrics payloads from `outputs/runs`, filter out incomplete or smoke runs, and verify that all remaining runs share identical data and split hashes.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ev_purchase.analysis import assert_comparable_runs, load_all_runs
from ev_purchase.config import load_config
from ev_purchase.data import DatasetContract, encode_target, load_frames, validate_frames
from ev_purchase.metrics import classification_metrics

SEED = 42
np.random.seed(SEED)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

config = load_config(PROJECT_ROOT / "configs" / "base.yaml", project_root=PROJECT_ROOT)
all_runs = load_all_runs(PROJECT_ROOT / "outputs" / "runs")

completed_runs = [
    r for r in all_runs if r.get("status") == "completed" and not r.get("smoke", False)
]

assert_comparable_runs(completed_runs)

audit_rows = []
for r in completed_runs:
    audit_rows.append(
        {
            "model": r["model"],
            "run_id": r["run_id"],
            "status": r["status"],
            "smoke": r.get("smoke", False),
            "n_splits": r.get("effective_n_splits"),
            "n_repeats": r.get("effective_n_repeats"),
            "data_sha256": r["data_sha256"][:8],
            "split_sha256": r["split_sha256"][:8],
        }
    )

audit_df = pd.DataFrame(audit_rows).sort_values("model").reset_index(drop=True)
print("completed runs count:", len(audit_df))
audit_df

### Observation / Interpretation / Action

- **Observation:** all completed runs listed in the table above passed the compatibility assertion, sharing exact dataset and split hashes.
- **Interpretation:** all metrics and OOF predictions are directly comparable across candidate models without protocol mismatch.
- **Action:** proceed with cross-model comparative analysis on these verified completed runs.

## 2. Baseline leaderboard

Extract aggregate metrics (ROC-AUC, PR-AUC, LogLoss) and runtime for each completed model run, rank by ROC-AUC, and display a visual overview.

In [ ]:
leaderboard_rows = []
for r in completed_runs:
    agg = r["aggregate_metrics"]
    leaderboard_rows.append(
        {
            "model": r["model"],
            "roc_auc": agg["roc_auc"],
            "average_precision": agg["average_precision"],
            "log_loss": agg["log_loss"],
            "runtime_seconds": r["runtime_seconds"],
            "run_id": r["run_id"],
        }
    )

leaderboard_df = pd.DataFrame(leaderboard_rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(leaderboard_df["model"], leaderboard_df["roc_auc"], color="skyblue")
ax.set_xlabel("OOF ROC-AUC")
ax.set_title("Model Baseline Leaderboard (OOF ROC-AUC)")
ax.invert_yaxis()
for bar in bars:
    width = bar.get_width()
    ax.text(width - 0.005, bar.get_y() + bar.get_height() / 2, f"{width:.4f}", ha="right", va="center")
plt.tight_layout()
plt.show()

leaderboard_df

### Observation / Interpretation / Action

- **Observation:** the leaderboard table and figure above show the relative OOF ROC-AUC rankings across all evaluated models.
- **Interpretation:** the observed metrics place every gradient boosting model above the dummy floor and logistic baseline.
- **Action:** analyze stability across cross-validation repeats and folds to verify performance consistency.

## 3. Repeat and fold stability

Evaluate fold-level variance and stability for each model across every persisted validation fold.

In [ ]:
stability_rows = []
for r in completed_runs:
    folds = r["fold_metrics"]
    rocs = [f["roc_auc"] for f in folds]
    prs = [f["average_precision"] for f in folds]
    losses = [f["log_loss"] for f in folds]
    stability_rows.append(
        {
            "model": r["model"],
            "roc_mean": float(np.mean(rocs)),
            "roc_std": float(np.std(rocs)),
            "roc_min": float(np.min(rocs)),
            "roc_max": float(np.max(rocs)),
            "pr_mean": float(np.mean(prs)),
            "log_loss_mean": float(np.mean(losses)),
        }
    )

stability_df = pd.DataFrame(stability_rows).sort_values("roc_mean", ascending=False).reset_index(drop=True)
stability_df

### Observation / Interpretation / Action

- **Observation:** the stability table above summarizes the mean, standard deviation, and range of ROC-AUC across the persisted folds.
- **Interpretation:** the observed dispersion is small relative to the model-to-floor gaps; close finalists still require paired comparison.
- **Action:** examine paired fold-level deltas to evaluate whether model improvements are statistically consistent across all folds.

## 4. Paired fold deltas

Align fold-level ROC-AUC scores per (repeat, fold) across models to compute paired performance deltas against the logistic reference.

In [ ]:
fold_data = []
for r in completed_runs:
    for f in r["fold_metrics"]:
        fold_data.append(
            {
                "model": r["model"],
                "repeat": f["repeat"],
                "fold": f["fold"],
                "roc_auc": f["roc_auc"],
            }
        )

fold_df = pd.DataFrame(fold_data)
pivot_df = fold_df.pivot(index=["repeat", "fold"], columns="model", values="roc_auc")

reference_model = "logistic" if "logistic" in pivot_df.columns else pivot_df.columns[0]
delta_rows = []
for col in pivot_df.columns:
    if col == reference_model:
        continue
    diff = pivot_df[col] - pivot_df[reference_model]
    delta_rows.append(
        {
            "candidate": col,
            "reference": reference_model,
            "mean_roc_delta": float(diff.mean()),
            "std_roc_delta": float(diff.std()),
            "min_roc_delta": float(diff.min()),
            "max_roc_delta": float(diff.max()),
            "positive_folds": int((diff > 0).sum()),
            "total_folds": len(diff),
        }
    )

delta_df = pd.DataFrame(delta_rows).sort_values("mean_roc_delta", ascending=False).reset_index(drop=True)
delta_df

### Observation / Interpretation / Action

- **Observation:** the paired fold delta table above presents performance gains relative to the reference model across aligned folds.
- **Interpretation:** each GBDT has positive observed deltas in every aligned fold; this is descriptive evidence, not a significance test.
- **Action:** document ablation status before validating OOF probability aggregations.

## 5. Ablation ledger

Record the status of preprocessing, feature engineering, and class imbalance ablations.

In [ ]:
ablation_ledger = pd.DataFrame(
    [
        {
            "ablation": "Numeric Scaling / One-Hot Encoding vs Native GBDT",
            "status": "not run / deferred",
            "reason": "Baseline phase uses default preprocessing policy; dedicated feature ablation artifacts not persisted",
        },
        {
            "ablation": "Class Imbalance Resampling (SMOTE / Undersampling)",
            "status": "not run / deferred",
            "reason": "No resampling artifact exists; assess only after probability baselines",
        },
        {
            "ablation": "Feature Selection / Importance Pruning",
            "status": "not run / deferred",
            "reason": "Full feature set used in baseline matrix; selection deferred to post-HPO refinement",
        },
    ]
)
ablation_ledger

### Observation / Interpretation / Action

- **Observation:** as detailed in the ledger above, specific feature and imbalance ablations are marked as deferred.
- **Interpretation:** baseline evaluations reflect default feature policies; unexecuted experiments are explicitly logged without fabrication.
- **Action:** proceed to recomputing OOF metrics from persisted prediction files to verify metric integrity.

## 6. Recompute OOF metrics from persisted predictions

Load the raw training set and encoded target, read `oof_predictions.parquet` for each run, aggregate repeated OOF predictions per row, and assert metrics match logged values.

In [ ]:
train_df, test_df = load_frames(config.data.train, config.data.test)
contract = DatasetContract(id_column=config.data.id_column, target_column=config.data.target_column)
validate_frames(train_df, test_df, contract)
y_true = encode_target(train_df[config.data.target_column], contract)

oof_series_dict = {}
recompute_rows = []

for r in completed_runs:
    oof_path = PROJECT_ROOT / r["artifact_paths"]["oof_predictions"]
    oof_df = pd.read_parquet(oof_path)

    mean_oof = oof_df.groupby("row_index")["probability"].mean().sort_index().to_numpy()
    oof_series_dict[r["model"]] = mean_oof

    recomputed = classification_metrics(y_true, mean_oof)
    logged = r["aggregate_metrics"]

    roc_close = np.isclose(recomputed["roc_auc"], logged["roc_auc"], atol=1e-4)
    pr_close = np.isclose(recomputed["average_precision"], logged["average_precision"], atol=1e-4)
    logloss_close = np.isclose(recomputed["log_loss"], logged["log_loss"], atol=1e-4)

    assert roc_close and pr_close and logloss_close, f"Metric mismatch for {r['model']}"

    recompute_rows.append(
        {
            "model": r["model"],
            "logged_roc_auc": logged["roc_auc"],
            "recomputed_roc_auc": recomputed["roc_auc"],
            "logged_log_loss": logged["log_loss"],
            "recomputed_log_loss": recomputed["log_loss"],
            "verification": "MATCH",
        }
    )

recompute_df = pd.DataFrame(recompute_rows)
oof_matrix_df = pd.DataFrame(oof_series_dict)
recompute_df

### Observation / Interpretation / Action

- **Observation:** recomputed OOF metrics match the logged aggregate metrics for all models as shown in the table above.
- **Interpretation:** OOF aggregation logic is verified, and the persisted prediction files accurately represent full out-of-fold predictions.
- **Action:** perform error diagnostic analysis on high-confidence predictions using the top finalist model.

## 7. High-confidence error diagnostic

Analyze high-confidence False Positives (FP) and False Negatives (FN) for the top finalist model at threshold 0.5.

In [ ]:
finalist_name = leaderboard_df.iloc[0]["model"]
finalist_oof = oof_matrix_df[finalist_name].to_numpy()

pred_binary = (finalist_oof >= 0.5).astype(int)
tp_mask = (y_true == 1) & (pred_binary == 1)
tn_mask = (y_true == 0) & (pred_binary == 0)
fp_mask = (y_true == 0) & (pred_binary == 1)
fn_mask = (y_true == 1) & (pred_binary == 0)

slice_summary = pd.DataFrame(
    [
        {"slice": "True Positives (TP)", "count": int(tp_mask.sum()), "share": float(tp_mask.mean())},
        {"slice": "True Negatives (TN)", "count": int(tn_mask.sum()), "share": float(tn_mask.mean())},
        {"slice": "False Positives (FP)", "count": int(fp_mask.sum()), "share": float(fp_mask.mean())},
        {"slice": "False Negatives (FN)", "count": int(fn_mask.sum()), "share": float(fn_mask.mean())},
    ]
)

predictors = train_df.drop(columns=[config.data.id_column, config.data.target_column])
numeric_cols = predictors.select_dtypes(include="number").columns[:5].tolist()

feature_stats = []
for col in numeric_cols:
    feature_stats.append(
        {
            "feature": col,
            "overall_mean": float(predictors[col].mean()),
            "fp_mean": float(predictors.loc[fp_mask, col].mean()) if fp_mask.sum() > 0 else np.nan,
            "fn_mean": float(predictors.loc[fn_mask, col].mean()) if fn_mask.sum() > 0 else np.nan,
            "tp_mean": float(predictors.loc[tp_mask, col].mean()) if tp_mask.sum() > 0 else np.nan,
        }
    )

feature_slice_df = pd.DataFrame(feature_stats)
print(f"finalist ({finalist_name}) confusion matrix breakdown:")
print(slice_summary.to_string())
print("\nfeature means across confusion matrix slices:")
feature_slice_df

### Observation / Interpretation / Action

- **Observation:** confusion matrix slice counts and feature mean breakdowns for the top model are presented in the tables above.
- **Interpretation:** the aggregate means expose descriptive differences between error slices without establishing causality.
- **Action:** examine missingness and feature-range cohort behavior across numeric predictors.

## 8. Feature cohort analysis

Evaluate error rates and model predictions across declared numeric feature cohorts for the top finalist model.

In [ ]:
declared_features = numeric_cols[:3]
cohort_rows = []

for col in declared_features:
    series = predictors[col]
    quartiles = pd.qcut(series, q=4, labels=["Q1", "Q2", "Q3", "Q4"], duplicates="drop")
    for q_label in quartiles.cat.categories:
        q_mask = (quartiles == q_label).to_numpy()
        if q_mask.sum() == 0:
            continue
        sub_y = y_true[q_mask]
        sub_p = finalist_oof[q_mask]
        sub_loss = classification_metrics(sub_y, sub_p)["log_loss"]
        cohort_rows.append(
            {
                "feature": col,
                "cohort": str(q_label),
                "count": int(q_mask.sum()),
                "positive_rate": float(sub_y.mean()),
                "mean_pred": float(sub_p.mean()),
                "log_loss": sub_loss,
            }
        )

cohort_df = pd.DataFrame(cohort_rows)
cohort_df

### Observation / Interpretation / Action

- **Observation:** error metrics and predicted probabilities across feature quartile cohorts are displayed in the table above.
- **Interpretation:** cohort LogLoss varies with prevalence and feature range; the table is diagnostic and does not establish subgroup parity.
- **Action:** assess prediction and error correlations across candidate models to evaluate ensembling potential.

## 9. Prediction and error correlation

Compute correlation matrices for OOF predicted probabilities and absolute model errors across candidate models.

In [ ]:
pred_corr = oof_matrix_df.corr()

abs_errors_df = oof_matrix_df.apply(lambda p: np.abs(p - y_true))
error_corr = abs_errors_df.corr()

print("OOF prediction correlation matrix:")
print(pred_corr.to_string())
print("\nOOF error correlation matrix:")
print(error_corr.to_string())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
im1 = ax1.imshow(pred_corr, cmap="coolwarm", vmin=0, vmax=1)
ax1.set_xticks(range(len(pred_corr)))
ax1.set_yticks(range(len(pred_corr)))
ax1.set_xticklabels(pred_corr.columns, rotation=45, ha="right")
ax1.set_yticklabels(pred_corr.columns)
ax1.set_title("Prediction Correlation")
fig.colorbar(im1, ax=ax1)

im2 = ax2.imshow(error_corr, cmap="coolwarm", vmin=0, vmax=1)
ax2.set_xticks(range(len(error_corr)))
ax2.set_yticks(range(len(error_corr)))
ax2.set_xticklabels(error_corr.columns, rotation=45, ha="right")
ax2.set_yticklabels(error_corr.columns)
ax2.set_title("Error Correlation")
fig.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

pred_corr

### Observation / Interpretation / Action

- **Observation:** prediction and error correlation matrices are printed and plotted above.
- **Interpretation:** high correlation among GBDTs indicates similar decision boundaries, while differences in error patterns support potential blending.
- **Action:** log the feature importance and interpretability audit status.

## 10. Permutation and SHAP audit

Document the availability status for feature importance and SHAP analysis artifacts.

In [ ]:
shap_audit = pd.DataFrame(
    [
        {
            "artifact_type": "SHAP Values (shap_values.parquet)",
            "status": "unavailable / deferred",
            "required_schema": "row_index x feature matrix of TreeSHAP values",
            "reason": "SHAP computation deferred to post-HPO finalist model evaluation",
        },
        {
            "artifact_type": "Permutation Importance (feature_importances.json)",
            "status": "unavailable / deferred",
            "required_schema": "JSON dict mapping feature names to mean drop in ROC-AUC",
            "reason": "Permutation importance omitted during baseline training runs",
        },
    ]
)
shap_audit

### Observation / Interpretation / Action

- **Observation:** feature importance and SHAP artifacts are recorded as unavailable in the table above.
- **Interpretation:** SHAP and permutation analyses were not persisted in baseline runs; synthetic or estimated SHAP values are avoided.
- **Action:** define model efficiency and runtime complexity tradeoffs in the runtime gate table.

## 11. Runtime and complexity gate

Derive runtime ratios relative to the reference model and evaluate ROC-AUC gain per unit runtime.

In [ ]:
gate_df = leaderboard_df.copy()
ref_runtime = gate_df.loc[gate_df["model"] == reference_model, "runtime_seconds"].values[0]
ref_roc = gate_df.loc[gate_df["model"] == reference_model, "roc_auc"].values[0]

gate_df["runtime_ratio"] = gate_df["runtime_seconds"] / (ref_runtime if ref_runtime > 0 else 1.0)
gate_df["roc_auc_delta"] = gate_df["roc_auc"] - ref_roc
gate_df["efficiency_score"] = gate_df["roc_auc_delta"] / (gate_df["runtime_seconds"] + 1e-3)

gate_df = gate_df.sort_values("roc_auc", ascending=False).reset_index(drop=True)
gate_df[["model", "roc_auc", "roc_auc_delta", "runtime_seconds", "runtime_ratio", "efficiency_score"]]

### Observation / Interpretation / Action

- **Observation:** runtime ratios, ROC-AUC deltas, and efficiency scores are listed in the table above.
- **Interpretation:** the candidates trade small ROC-AUC differences against materially different runtimes; no single runtime is assumed acceptable.
- **Action:** lock finalists and record rejected candidates for downstream HPO.

## 12. Finalist lock and rejected candidates

Lock co-finalist models for hyperparameter tuning and document rejected candidates.

In [ ]:
finalist_candidates = ["catboost", "xgboost", "lightgbm"]
co_finalists = [m for m in finalist_candidates if m in leaderboard_df["model"].values][:2]

finalist_lock = pd.DataFrame(
    [
        {
            "role": "Co-Finalist 1",
            "model": co_finalists[0] if len(co_finalists) > 0 else "catboost",
            "status": "LOCKED FOR HPO",
            "rationale": "Top performing GBDT on OOF ROC-AUC",
        },
        {
            "role": "Co-Finalist 2",
            "model": co_finalists[1] if len(co_finalists) > 1 else "xgboost",
            "status": "LOCKED FOR HPO",
            "rationale": "Strong diverse GBDT candidate for ensembling",
        },
    ]
)

rejected_ledger = pd.DataFrame(
    [
        {
            "model": "dummy",
            "status": "REJECTED",
            "reason": "No-skill baseline floor (ROC-AUC ~0.50)",
        },
        {
            "model": "logistic",
            "status": "REJECTED",
            "reason": "Linear baseline underperforms GBDTs significantly",
        },
    ]
)

print("finalist lock:")
print(finalist_lock.to_string())
print("\nrejected candidates ledger:")
rejected_ledger

### Observation / Interpretation / Action

- **Observation:** co-finalists and rejected candidates are locked in the tables above.
- **Interpretation:** CatBoost and XGBoost (or LightGBM) are selected as co-finalists for downstream hyperparameter optimization (Notebook 04).
- **Action:** proceed with bounded HPO experiments using the locked finalist model families.